In [ ]:
# Update yt-dlp to the latest nightly build to fix YouTube download errors
!pip install -U yt-dlp
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
from datasets import load_dataset
import yt_dlp
import os

# 1. Configuration
TARGET_COUNT = 100  # How many English clips you want
OUTPUT_DIR = "./talkvid_english_subset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Load Dataset (Switching to 'train' split is usually better for finding enough samples)
print("Loading TalkVid metadata...")
ds = load_dataset("FreedomIntelligence/TalkVid", split="test", streaming=True)

# 3. Processing Function
def process_downloads():
    downloaded_count = 0

    # Configure downloader options
    ydl_opts = {
        'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': f'{OUTPUT_DIR}/%(id)s.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        # Force overwrite to ensure clean files
        'overwrites': True,
        # This downloads only the specific time range
        'download_ranges': lambda info, ydl: [
            {'start_time': item['start'], 'end_time': item['end']}
        ],
        'force_keyframes_at_cuts': True,
    }

    print(f"Searching for {TARGET_COUNT} English clips...")

    for i, sample in enumerate(ds):
        if downloaded_count >= TARGET_COUNT:
            break

        # --- EXTRACT INFO ---
        info = sample.get("info", {})
        language = info.get("Language", "Unknown")
        video_url = info.get("Video Link") or sample.get("video_url")
        start = sample.get("start-time")
        end = sample.get("end-time")
        clip_id = sample.get("id")

        # --- FILTER: ENGLISH ONLY ---
        if language != "English":
            continue

        # Check validity
        if not video_url or start is None or end is None:
            continue

        # Prepare item dictionary for the downloader
        item = {'url': video_url, 'start': start, 'end': end, 'id': clip_id}

        # --- DOWNLOAD ---
        try:
            # We initialize the downloader fresh for the specific range callback
            current_opts = ydl_opts.copy()
            current_opts['download_ranges'] = yt_dlp.utils.download_range_func(None, [(start, end)])

            # Explicitly set the output filename to the ID provided in dataset
            current_opts['outtmpl'] = f'{OUTPUT_DIR}/{clip_id}.%(ext)s'

            with yt_dlp.YoutubeDL(current_opts) as ydl:
                ydl.download([video_url])

            print(f"[{downloaded_count + 1}/{TARGET_COUNT}] ✅ {clip_id} ({start:.1f}s-{end:.1f}s)")
            downloaded_count += 1

        except Exception as e:
            # If a video is unavailable/private, we just skip it and keep looking
            # We don't print the full error to keep the log clean, just the ID
            print(f"⚠️ Skipped {clip_id}: Video unavailable or download failed.")
            continue

process_downloads()
print(f"\n Finished! English clips saved to {OUTPUT_DIR}")

Loading TalkVid metadata...
Searching for 100 English clips...
[1/100] ✅ videovideo-hOQAsNg8aw-scene3-scene3 (56.9s-62.0s)
[2/100] ✅ videovideo2482_4v01hRqzlQg-scene69-scene9 (1918.5s-1923.5s)
[3/100] ✅ videovideo2525_H3yZUfZV9UM-scene35-scene10 (844.0s-849.0s)
[4/100] ✅ videovideo2535_y3-nmMtxtYM-scene5-scene3 (39.5s-44.6s)
[5/100] ✅ videovideo6-198fR9Sxo-scene61-scene2 (625.2s-630.2s)
[6/100] ✅ videovideo71oNb0HPf5E-scene27-scene3 (470.8s-475.9s)
[7/100] ✅ videovideo8351_PvPwzmobAJw-scene11-scene5 (294.5s-299.6s)
[8/100] ✅ videovideo8517_cRM7G7wjqp0-scene13-scene3 (613.9s-618.9s)




ERROR: ffmpeg exited with code 1


⚠️ Skipped videovideo8JsY4OJMoqs-scene80-scene12: Video unavailable or download failed.
[9/100] ✅ videovideoAEKXLkRMB10-scene9-scene2 (63.5s-68.6s)
[10/100] ✅ videovideoAl5MFMZvpY4-scene2-scene1 (13.1s-18.1s)
[11/100] ✅ videovideoAl5MFMZvpY4-scene2-scene3 (23.3s-28.3s)
[12/100] ✅ videovideoCeULH0LrXh8-scene150-scene1 (853.6s-858.6s)
[13/100] ✅ videovideoGBugVPs27Os-scene38-scene1 (240.0s-245.0s)
[14/100] ✅ videovideoGBugVPs27Os-scene38-scene3 (255.1s-260.1s)




ERROR: ffmpeg exited with code 1


⚠️ Skipped videovideoGBugVPs27Os-scene38-scene6: Video unavailable or download failed.
[15/100] ✅ videovideoGOqEl4ADyVk-scene78-scene2 (457.6s-462.7s)
[16/100] ✅ videovideoKAvFxulDs2o-scene6-scene2 (131.5s-136.5s)


...

⚠️ Skipped videovideo_vMWyVHKDdtQ-scene7-scene22: Video unavailable or download failed.




ERROR: ffmpeg exited with code 1


⚠️ Skipped videovideoigDXJuTVdyI-scene6-scene58: Video unavailable or download failed.
[84/100] ✅ videovideojqigWVI2ves-scene1-scene4 (30.4s-35.4s)




ERROR: ffmpeg exited with code 1


⚠️ Skipped videovideoqIOoX85v7vs-scene51-scene17: Video unavailable or download failed.
[85/100] ✅ videovideoqzawsRCwvj8-scene47-scene1 (263.7s-268.7s)

🎉 Finished! English clips saved to ./talkvid_english_subset
